# Day 06: Production LLM API engineering

A model call is one line. A production LLM app is everything around it. This notebook demonstrates topics 6.2 to 6.10 with real API calls (6.1 is the `01_architecture/` app, and SSE is in `02_streaming_sse/`).

| § | Topic |
|---|---|
| 2 | Stateless vs stateful |
| 3 | Conversation history and context management |
| 4 | Streaming vs non-streaming |
| 5 | Error handling and API failures |
| 6 | Timeouts, retries, exponential backoff |
| 7 | Rate limits and quotas |
| 8 | Token usage and cost tracking |
| 9 | Latency: TTFT vs total time, P50 / P95 |
| 10 | Model selection and provider fallback |

Needs `OPENAI_API_KEY` in `day-06/code/.env`. A full run costs well under $0.02.

In [1]:
import asyncio, random, time
from dotenv import load_dotenv
import openai
from openai import OpenAI, AsyncOpenAI
import tiktoken

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

# USD per 1M tokens, from developers.openai.com/api/docs/models (checked 2026-10-16)
PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5-mini":  {"input": 0.25, "output": 2.00},
}

def cost_usd(model, input_tokens, output_tokens):
    p = PRICES[model]
    return input_tokens / 1e6 * p["input"] + output_tokens / 1e6 * p["output"]

def ask(input, model=MODEL, instructions=None, **kwargs):
    """One non-streaming call. Returns the text plus what production code should log."""
    start = time.perf_counter()
    r = client.responses.create(model=model, instructions=instructions, input=input, **kwargs)
    ms = round((time.perf_counter() - start) * 1000)
    u = r.usage
    return {"text": r.output_text, "in": u.input_tokens, "out": u.output_tokens, "ms": ms,
            "usd": cost_usd(model, u.input_tokens, u.output_tokens), "id": r.id}

## 2. Stateless vs stateful

The model API is **stateless**: every call starts from zero. Memory is something *your app* builds, by sending history again (or asking the provider to keep it).

In [2]:
# Two separate calls, no history: the second call knows nothing about the first
print(ask("My name is Sarthak.")["text"])
print("---")
print(ask("What is my name?")["text"])

Nice to meet you, Sarthak! How can I assist you today?
---


I don't know your name unless you tell me. Would you like to share it?


In [3]:
# Option A: the CLIENT/BACKEND keeps the history and sends it every time
history = [
    {"role": "user", "content": "My name is Sarthak."},
    {"role": "assistant", "content": "Nice to meet you, Sarthak!"},
    {"role": "user", "content": "What is my name?"},
]
print(ask(history)["text"])

Your name is Sarthak.


In [4]:
# Option B: the PROVIDER keeps the state. The Responses API can chain calls with previous_response_id
first = client.responses.create(model=MODEL, input="My name is Sarthak.")
second = client.responses.create(model=MODEL, input="What is my name?", previous_response_id=first.id)
print(second.output_text)
print("input tokens billed on call 2:", second.usage.input_tokens, "<- the earlier turn is still sent to the model, and billed")

Your name is Sarthak.
input tokens billed on call 2: 43 <- the earlier turn is still sent to the model, and billed


**Takeaway:** "stateful" only means *someone* stores the conversation: your DB, your client, or the provider. The model itself never remembers. Even with `previous_response_id`, the old turns are re-read and billed as input tokens.

## 3. Conversation history and context management

Every turn resends the whole history, so input tokens (and cost) grow with each message. Three strategies: send everything, a sliding window, or summary + window.

In [5]:
enc = tiktoken.get_encoding("o200k_base")  # the tokenizer gpt-4o-mini uses
def count_tokens(messages):
    return sum(len(enc.encode(m["content"])) + 4 for m in messages)  # ~4 tokens overhead per message

# A 40-turn support chat. Turn 1 holds a fact we'll need later.
convo = [{"role": "user", "content": "Hi, I'm planning a trip to Japan in April. My total budget is 2 lakh rupees."},
         {"role": "assistant", "content": "Great choice! April is cherry blossom season. How can I help you plan?"}]
topics = ["flights from Delhi", "hotels in Tokyo", "the JR Pass", "visa documents", "Kyoto temples", "food to try",
          "a SIM card", "packing for spring", "day trips from Osaka", "travel insurance"]
for i in range(39):
    t = topics[i % len(topics)]
    convo.append({"role": "user", "content": f"Can you tell me more about {t}? Give me the key points."})
    convo.append({"role": "assistant", "content": f"Here are the key points about {t}: " + "useful detail, " * 25})

WINDOW = 6  # keep the last 6 messages
full, window = [], []
for turn in range(1, 41):
    msgs = convo[: turn * 2]
    full.append(count_tokens(msgs))
    window.append(count_tokens(msgs[-WINDOW:]))

for turn in [1, 10, 20, 30, 40]:
    print(f"turn {turn:>2}: full history {full[turn-1]:>5} tokens | sliding window {window[turn-1]:>4} tokens")
print(f"\nTotal input tokens over 40 turns: full {sum(full):,} vs window {sum(window):,}")
print(f"Cost of history alone (gpt-4o-mini): full ${cost_usd(MODEL, sum(full), 0):.4f} vs window ${cost_usd(MODEL, sum(window), 0):.4f} per conversation")

turn  1: full history    45 tokens | sliding window   45 tokens
turn 10: full history  1033 tokens | sliding window  332 tokens
turn 20: full history  2129 tokens | sliding window  332 tokens
turn 30: full history  3225 tokens | sliding window  332 tokens
turn 40: full history  4321 tokens | sliding window  332 tokens

Total input tokens over 40 turns: full 87,280 vs window 12,629
Cost of history alone (gpt-4o-mini): full $0.0131 vs window $0.0019 per conversation


In [6]:
# The catch: a sliding window FORGETS. Ask about the budget from turn 1
question = {"role": "user", "content": "What was my total budget again?"}
print("WINDOW ONLY:", ask(convo[-WINDOW:] + [question])["text"])

WINDOW ONLY: I’m unable to access your previous conversations or documents. If you could provide more details about your budget or context, I'd be happy to help you calculate it or provide a general guide!


In [7]:
# Fix: summarise the old turns once, keep the summary + the recent window
old_turns = convo[:-WINDOW]
# Pass the old turns as ONE transcript. Passed as chat messages, the model just answers the last question in them
transcript = "\n".join(f"{m['role']}: {m['content']}" for m in old_turns)
summary = ask(transcript, instructions="Summarise this conversation in 2 sentences. Keep every number and fact the user stated.")
print("SUMMARY:", summary["text"], f"({summary['out']} tokens)\n")

summary_msg = {"role": "user", "content": "Summary of our earlier conversation: " + summary["text"]}
r = ask([summary_msg] + convo[-WINDOW:] + [question])
print("SUMMARY + WINDOW:", r["text"])
print(f"input tokens: {r['in']} (vs {count_tokens(convo + [question])} for the full history)")

SUMMARY: The user is planning a trip to Japan in April with a total budget of 2 lakh rupees. They inquired about various aspects of the trip, including flights from Delhi, hotels in Tokyo, the JR Pass, visa documents, Kyoto temples, food to try, SIM cards, spring packing tips, day trips from Osaka, and travel insurance. (71 tokens)



SUMMARY + WINDOW: Your total budget for the trip to Japan is 2 lakh rupees.
input tokens: 426 (vs 4332 for the full history)


**Takeaway:** context management = choosing what goes in the window. Sliding window is cheap but forgets; summary + window keeps facts at a fraction of the tokens. Later weeks add two more: **persistent memory** (week 3) and **retrieval** (RAG, week 2).

## 4. Streaming vs non-streaming

Same prompt, two ways. Watch *when the first words arrive*.

In [8]:
PROMPT = "Explain RAG to a web developer in about 150 words."

# Non-streaming: wait for everything
start = time.perf_counter()
r = client.responses.create(model=MODEL, input=PROMPT)
print(f"NON-STREAMING: nothing for {time.perf_counter() - start:.2f}s, then all {r.usage.output_tokens} tokens at once")

NON-STREAMING: nothing for 2.10s, then all 169 tokens at once


In [9]:
# Streaming: chunks arrive as they are generated
start = time.perf_counter()
ttft, n_chunks = None, 0
stream = client.responses.create(model=MODEL, input=PROMPT, stream=True)
for event in stream:
    if event.type == "response.output_text.delta":
        n_chunks += 1
        if ttft is None:
            ttft = time.perf_counter() - start
        if n_chunks <= 8:
            print(f"  {time.perf_counter() - start:5.2f}s  {event.delta!r}")
    elif event.type == "response.completed":
        total = time.perf_counter() - start
        usage = event.response.usage
print(f"  ... {n_chunks} chunks in total")
print(f"STREAMING: first words after {ttft:.2f}s, finished after {total:.2f}s ({usage.output_tokens} tokens)")

   0.96s  'R'
   0.96s  'AG'
   0.96s  ','
   0.96s  ' or'
   0.96s  ' Retrieval'
   1.12s  '-Aug'
   1.12s  'mented'
   1.12s  ' Generation'


  ... 170 chunks in total
STREAMING: first words after 0.96s, finished after 2.27s (171 tokens)


**Takeaway:** streaming doesn't make the model faster: total time is about the same. It makes the app *feel* faster, because the user starts reading after a fraction of a second. To get this into a browser, the backend forwards each chunk, e.g. as **Server-Sent Events**: see `02_streaming_sse/`.

## 5. Error handling and API failures

Trigger real errors on purpose and look at what the SDK raises.

In [10]:
def try_call(label, fn):
    try:
        fn()
        print(f"{label:<28} OK")
    except openai.APIStatusError as e:
        print(f"{label:<28} {e.status_code} {type(e).__name__}")
    except openai.APIConnectionError as e:
        print(f"{label:<28} --- {type(e).__name__}")

bad_key = OpenAI(api_key="sk-wrong-key", max_retries=0)
try_call("wrong API key",       lambda: bad_key.responses.create(model=MODEL, input="hi"))
try_call("model doesn't exist", lambda: client.responses.create(model="gpt-does-not-exist", input="hi"))
try_call("invalid parameter",   lambda: client.responses.create(model=MODEL, input="hi", temperature=5))
try_call("unreachable server",  lambda: OpenAI(base_url="http://127.0.0.1:9", max_retries=0).responses.create(model=MODEL, input="hi"))

wrong API key                401 AuthenticationError


model doesn't exist          404 NotFoundError


invalid parameter            400 BadRequestError
unreachable server           --- APIConnectionError


In [11]:
# Classify: is it worth retrying? (the same rules the OpenAI SDK uses internally)
RETRYABLE_STATUS = {408, 409, 429, 500, 502, 503, 504}

def is_retryable(e: Exception) -> bool:
    if isinstance(e, (openai.APITimeoutError, openai.APIConnectionError)):
        return True                                  # network blip / timeout: temporary
    if isinstance(e, openai.APIStatusError):
        return e.status_code in RETRYABLE_STATUS     # 400/401/403/404: your bug, retrying won't help
    return False

def user_message(e: Exception) -> str:
    """What the USER sees. Never the raw error or a stack trace."""
    if isinstance(e, openai.RateLimitError):
        return "We're getting a lot of requests. Please try again in a moment."
    if is_retryable(e):
        return "The AI service is temporarily unavailable. Please try again."
    return "Something went wrong on our side. We've been notified."  # 401/400: alert the team, not the user

try:
    bad_key.responses.create(model=MODEL, input="hi")
except Exception as e:
    print("retryable?   ", is_retryable(e))
    print("user sees:   ", user_message(e))
    print("we log:      ", type(e).__name__, getattr(e, "status_code", None), "request_id:", getattr(e, "request_id", None))

retryable?    False
user sees:    Something went wrong on our side. We've been notified.
we log:       AuthenticationError 401 request_id: req_513f8f48f7044dfba8ae608408498808


## 6. Timeouts, retries, exponential backoff

**Timeout** = how long you wait. **Retry** = try again after a temporary failure. **Backoff** = wait longer after each failure.

In [12]:
# SDK defaults (openai 3.31): timeout 600s (10 minutes!), max_retries=2 with backoff.
print("default timeout:", client.timeout)
print("default max_retries:", client.max_retries)

# A 10-minute wait is far too long for a chat UI. Set your own:
impatient = OpenAI(timeout=0.05, max_retries=0)  # 50 ms: guaranteed to time out
start = time.perf_counter()
try:
    impatient.responses.create(model=MODEL, input="Write a long story.")
except openai.APITimeoutError as e:
    print(f"\n{type(e).__name__} after {time.perf_counter() - start:.2f}s")

default timeout: Timeout(connect=5.0, read=600, write=600, pool=600)
default max_retries: 2

APITimeoutError after 0.13s


In [13]:
# Write the retry loop yourself once, to see what the SDK does for you.
# A fake API that fails twice with 503, then succeeds:
class FlakyAPI:
    def __init__(self, fail_times): self.calls, self.fail_times = 0, fail_times
    def call(self):
        self.calls += 1
        if self.calls <= self.fail_times:
            raise RuntimeError("503 Service Unavailable")
        return "success"

def with_backoff(fn, max_retries=4, base=0.5, cap=8.0):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except RuntimeError as e:
            if attempt == max_retries:
                raise
            delay = min(cap, base * 2 ** attempt)        # 0.5, 1, 2, 4 ... capped at 8
            delay = delay * random.uniform(0.75, 1.0)    # jitter: don't retry in sync with everyone else
            print(f"  attempt {attempt + 1} failed ({e}); waiting {delay:.2f}s")
            time.sleep(delay)

random.seed(6)
api = FlakyAPI(fail_times=3)
print("result:", with_backoff(api.call), f"after {api.calls} calls")

  attempt 1 failed (503 Service Unavailable); waiting 0.47s


  attempt 2 failed (503 Service Unavailable); waiting 0.96s


  attempt 3 failed (503 Service Unavailable); waiting 1.74s


result: success after 4 calls


**Takeaway:** set a timeout that fits your UX, cap retries, back off exponentially with jitter, honour `Retry-After` on a 429, and **never retry a 400/401**. With the OpenAI SDK, that is `OpenAI(timeout=20, max_retries=3)`. Note a timeout stops *your* waiting; the provider may still finish (and bill) the request.

## 7. Rate limits and quotas

**Rate limit** = how fast (requests or tokens per minute). **Quota** = how much in total (credits, a budget).

In [14]:
# The provider tells you where you stand in the response headers of every call
raw = client.responses.with_raw_response.create(model=MODEL, input="hi")
for h in ["x-ratelimit-limit-requests", "x-ratelimit-remaining-requests",
          "x-ratelimit-limit-tokens", "x-ratelimit-remaining-tokens", "x-ratelimit-reset-tokens"]:
    print(f"{h:<34} {raw.headers.get(h)}")

x-ratelimit-limit-requests         30000
x-ratelimit-remaining-requests     29999
x-ratelimit-limit-tokens           150000000
x-ratelimit-remaining-tokens       149999975
x-ratelimit-reset-tokens           0s


In [15]:
# RPM vs TPM: you can hit the token limit long before the request limit
rpm_limit, tpm_limit = 60, 10_000            # example limits, not real ones
requests_per_min, tokens_per_request = 60, 500
tpm = requests_per_min * tokens_per_request
print(f"{requests_per_min} req/min is within {rpm_limit} RPM, but uses {tpm:,} tokens/min vs a {tpm_limit:,} TPM limit -> 429s")

60 req/min is within 60 RPM, but uses 30,000 tokens/min vs a 10,000 TPM limit -> 429s


In [16]:
# Protect yourself: cap how many calls run at once with a semaphore
aclient = AsyncOpenAI()
in_flight = peak = 0

async def limited_call(sem, i):
    global in_flight, peak
    async with sem:
        in_flight += 1; peak = max(peak, in_flight)
        r = await aclient.responses.create(model=MODEL, input=f"Say the number {i} in French. One word.")
        in_flight -= 1
        return r.output_text

sem = asyncio.Semaphore(3)
start = time.perf_counter()
answers = await asyncio.gather(*(limited_call(sem, i) for i in range(1, 11)))
print(answers)
print(f"10 calls, at most {peak} at the same time, {time.perf_counter() - start:.1f}s")

['Un.', 'Deux.', 'Trois.', 'Quatre.', 'Cinq.', 'Six', 'Sept.', 'Huit.', 'Neuf.', 'Dix.']
10 calls, at most 3 at the same time, 4.3s


In [17]:
# Quota: a per-user daily budget your backend enforces BEFORE calling the model
DAILY_BUDGET_USD = 0.0001
spent = {"saurabh": 0.0}

def budgeted_ask(user, prompt):
    if spent[user] >= DAILY_BUDGET_USD:
        return f"[blocked] {user} has used ${spent[user]:.6f} of ${DAILY_BUDGET_USD} today"
    r = ask(prompt)
    spent[user] += r["usd"]
    return f"${spent[user]:.6f} spent so far | {r['text'][:50]}..."

for _ in range(6):
    print(budgeted_ask("saurabh", "Give me one tip for writing clean code, in 30 words."))

$0.000025 spent so far | Use meaningful variable names that clearly describ...


$0.000049 spent so far | Use meaningful and descriptive names for variables...


$0.000075 spent so far | Write meaningful variable and function names that ...


$0.000102 spent so far | Adopt meaningful naming conventions for variables,...
[blocked] saurabh has used $0.000102 of $0.0001 today
[blocked] saurabh has used $0.000102 of $0.0001 today


## 8. Token usage and cost tracking

Every response includes `usage`. Log it on every call, with the model and the request ID.

In [18]:
r = client.responses.create(model=MODEL, input="Explain what a token is, in 2 sentences.")
u = r.usage
print("input tokens: ", u.input_tokens)
print("output tokens:", u.output_tokens)
print(f"cost:          ${cost_usd(MODEL, u.input_tokens, u.output_tokens):.6f}")
print(f"x 1M requests: ${cost_usd(MODEL, u.input_tokens, u.output_tokens) * 1e6:,.0f}")

input tokens:  18
output tokens: 50
cost:          $0.000033
x 1M requests: $33


In [19]:
# The summary's worked example: $2 / $8 per 1M tokens, 10,000 in + 2,000 out
print(f"${10_000/1e6*2:.3f} + ${2_000/1e6*8:.3f} = ${10_000/1e6*2 + 2_000/1e6*8:.3f} per request")

$0.020 + $0.016 = $0.036 per request


In [20]:
# Hidden tokens: reasoning models think before answering, and thinking is billed as OUTPUT
r = client.responses.create(model="gpt-5-mini", input="Is 391 a prime number? Answer yes or no.")
u = r.usage
print("answer:", r.output_text)
print("output tokens:", u.output_tokens, "of which reasoning:", u.output_tokens_details.reasoning_tokens)
print(f"cost: ${cost_usd('gpt-5-mini', u.input_tokens, u.output_tokens):.6f} for a one-word answer")

answer: No.
output tokens: 96 of which reasoning: 64
cost: $0.000197 for a one-word answer


## 9. Latency: TTFT vs total time, and P50 / P95

TTFT (time to first token) is what the user *feels*. Total time depends mostly on answer length.

In [21]:
def timed_stream(prompt, model=MODEL):
    start = time.perf_counter(); ttft = None
    for event in client.responses.create(model=model, input=prompt, stream=True):
        if event.type == "response.output_text.delta" and ttft is None:
            ttft = time.perf_counter() - start
        if event.type == "response.completed":
            return ttft, time.perf_counter() - start, event.response.usage.output_tokens

for label, prompt in [("short answer", "What is RAG? One sentence."),
                      ("long answer ", "Explain RAG in detail, about 300 words.")]:
    ttft, total, out = timed_stream(prompt)
    print(f"{label}: TTFT {ttft:.2f}s | total {total:.2f}s | {out} tokens")

short answer: TTFT 0.90s | total 1.07s | 30 tokens


long answer : TTFT 0.93s | total 3.82s | 390 tokens


In [22]:
# One measurement means nothing. Run it 10 times and look at the spread
ttfts, totals = [], []
for _ in range(10):
    ttft, total, _ = timed_stream("What is RAG? One sentence.")
    ttfts.append(ttft); totals.append(total)

def pct(xs, p):
    xs = sorted(xs); return xs[min(len(xs) - 1, round(p / 100 * (len(xs) - 1)))]

print("TTFT  (s):", [round(x, 2) for x in sorted(ttfts)])
print(f"TTFT  P50 {pct(ttfts, 50):.2f}s | P95 {pct(ttfts, 95):.2f}s")
print(f"Total P50 {pct(totals, 50):.2f}s | P95 {pct(totals, 95):.2f}s")

TTFT  (s): [0.86, 0.93, 0.95, 0.99, 1.08, 1.1, 1.37, 1.5, 1.5, 1.7]
TTFT  P50 1.08s | P95 1.70s
Total P50 1.28s | P95 2.80s


**Takeaway:** report P50 *and* P95. The average hides the slow requests that make users leave. (With 10 samples P95 is roughly the slowest run; in production you'd use thousands.)

## 10. Model selection and provider fallback

**Selection:** test candidates on *your* task and compare quality, latency and cost.

In [23]:
TICKETS = [
    ("I was charged twice for my order", "billing"),
    ("The app crashes when I upload a photo", "bug"),
    ("How do I change my delivery address?", "account"),
    ("Refund still not received after 10 days", "billing"),
    ("Login button does nothing on Safari", "bug"),
    ("Can I add a second email to my profile?", "account"),
]
INSTR = "Classify the support ticket as exactly one word: billing, bug, or account."

for model in ["gpt-4o-mini", "gpt-5-mini"]:
    correct, ms, usd = 0, 0, 0.0
    extra = {"reasoning": {"effort": "minimal"}} if model == "gpt-5-mini" else {}
    for text, label in TICKETS:
        r = ask(text, model=model, instructions=INSTR, **extra)
        correct += r["text"].strip().lower().strip(".") == label
        ms += r["ms"]; usd += r["usd"]
    print(f"{model:<12} accuracy {correct}/{len(TICKETS)} | avg {ms // len(TICKETS)} ms | ${usd:.6f} total")

gpt-4o-mini  accuracy 6/6 | avg 1337 ms | $0.000039 total


gpt-5-mini   accuracy 6/6 | avg 1080 ms | $0.000280 total


**Fallback:** if the primary fails with a *temporary* error, try a compatible backup. A permanent error (bad key, bad request) should fail loudly instead.

In [24]:
def ask_with_fallback(prompt, chain):
    """chain = [(name, client, model), ...] tried in order."""
    for name, c, model in chain:
        try:
            r = c.responses.create(model=model, input=prompt)
            return f"answered by {name}: {r.output_text[:60]}"
        except Exception as e:
            if not is_retryable(e):
                raise                                  # 401 / 400: fix it, don't hide it
            print(f"  {name} failed ({type(e).__name__}), falling back")
    return "All providers failed. Please try again later."

primary_down = OpenAI(timeout=0.01, max_retries=0)    # simulates an outage: always times out
chain = [("primary gpt-5-mini", primary_down, "gpt-5-mini"),
         ("backup gpt-4o-mini", client, "gpt-4o-mini")]
print(ask_with_fallback("What is RAG? One sentence.", chain))

  primary gpt-5-mini failed (APITimeoutError), falling back


answered by backup gpt-4o-mini: RAG stands for "Red, Amber, Green," a color-coded system use


In [25]:
# Same chain, but the primary has a WRONG KEY: no fallback, the error surfaces
chain = [("primary (bad key)", OpenAI(api_key="sk-wrong", max_retries=0), "gpt-5-mini"),
         ("backup gpt-4o-mini", client, "gpt-4o-mini")]
try:
    ask_with_fallback("What is RAG?", chain)
except openai.AuthenticationError as e:
    print("Raised, not hidden:", type(e).__name__, e.status_code)

Raised, not hidden: AuthenticationError 401


**Takeaway:** a backup must support what you need (context length, tools, structured output, data rules). In practice the backup is often a *different provider*, through a router such as OpenRouter or LiteLLM.

## Recap

A production LLM app is not a model call. It's a system that manages **context, failures, speed, usage, cost and reliability**. Day 07: how to *see* all of this per request (tracing) and how to know the answers are good (evals).